# 3 · Claude Code II: Debugging with AI & the Earnings Engine

**Outcome of this session:** two skills that make AI usable on a finance desk. First, you direct Claude to repair a broken valuation model, one failing test at a time. Second, you build the verification layer that machine-checks an AI-written analysis against its source.

**In this notebook you will:**

- Read a traceback and direct Claude to repair a broken DCF, one test at a time
- Validate structured model output with Pydantic
- Build the verification layer that machine-checks every quoted claim
- Generate an evidence-verified earnings memo from a transcript


## The debugging protocol

1. **Read the traceback bottom-up**: the last line says *what* happened, the marked line says *where*.
2. **Reproduce the error** before changing anything.
3. **Diagnose before fixing**: make Claude explain the cause first.
4. **Change one thing at a time.**

> A crash announces itself and forces a fix. The dangerous error is the one that returns a plausible but wrong number.

**Tests are the contract that lets you trust AI-written code.** Each test encodes one financial rule; when Claude edits the code, the tests decide whether the edit was right, so acceptance depends on verification rather than on how confident the model sounds.

In [ ]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# If you pulled course updates while this kernel was running, pick them up here
# (a no-op on a fresh kernel; saves a restart otherwise):
import importlib
for _n in [n for n in list(sys.modules) if n.startswith("toolkit")]:
    importlib.reload(sys.modules[_n])
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"repo root: {ROOT}")
print(f"API key:   {'configured' if HAS_KEY else 'NOT SET - cells that call Claude will be skipped'}")

## Part A: the broken DCF

`session-03-debugging/demo/broken_dcf.py` contains a **discounted cash flow (DCF) valuation** of Meridian Semiconductor, a fictional company trading at $62. A DCF estimates what a company is worth today by projecting its future free cash flows and discounting them back to the present. This copy ships with six planted bugs. You will not fix them by hand: your job in Lab 1 is to **direct Claude to fix them**, and to judge its work. First, run it and read the failure:

In [ ]:
import subprocess
print("EXPECTED FAILURE - broken_dcf.py is deliberately broken; you will fix it in the lab.")
print("=" * 78)
r = subprocess.run([sys.executable, str(ROOT / "session-03-debugging" / "demo" / "broken_dcf.py")],
                   capture_output=True, text=True)
print(r.stdout[-300:] if r.stdout else "", r.stderr[-500:])
print("=" * 78)
print("Diagnosis, bottom-up: WHAT = the last line (IndexError: the growth-rate list")
print("ran out). WHERE = the marked line (project_fcf, line 42: the loop assumes")
print("more years than the list contains - a hardcoded horizon). This is planted")
print("bug 1 of 6, and the only one that announces itself.")

In [ ]:
# and watch its sanity tests fail (7 failures = 7 pieces of violated financial logic)
r = subprocess.run([sys.executable, "-m", "pytest", str(ROOT / "session-03-debugging" / "demo"), "-q"],
                   capture_output=True, text=True)
print(r.stdout[-600:])

## Part B · Lab 1: make Claude repair it

The seven failing tests are the specification: each one names a financial rule the code violates. Your loop, repeated until all seven pass:

1. Run the ✅ check cell below and read the **first failing test's name**.
2. Open `session-03-debugging/demo/broken_dcf.py`, select the function that test targets, and press `Option+K` (`Alt+K` on Windows).
3. Ask the panel, naming the test: *"`test_wacc_after_tax_debt` fails. Show me the offending line, explain the financial error, and fix only that."*
4. **Read the diff before accepting.** The explanation must name a finance mistake, not just a code change.
5. Save the file, rerun the check cell, and take the next failing test.

Two rules of engagement: **one test at a time** (a wholesale rewrite teaches you nothing and hides new errors), and **never accept a diff you cannot explain**. If the panel proposes changing the test instead of the code, refuse: the tests are the contract.

In [ ]:
# ✅ check: run me after every fix
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", str(ROOT / "session-03-debugging" / "demo"), "-q"],
                   capture_output=True, text=True)
summary = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-200:]
print("pytest:", summary)

if "7 passed" in r.stdout:
    v = subprocess.run([sys.executable, str(ROOT / "session-03-debugging" / "demo" / "broken_dcf.py")],
                       capture_output=True, text=True)
    print(v.stdout.strip().splitlines()[-1] if v.stdout.strip() else v.stderr[-200:])
    assert "75.61" in v.stdout, "tests pass but the valuation is off - compare with the test file"
    print("\nAll seven rules hold, and the model values Meridian at $75.61 per share")
    print("against a $62 market price. Now the investment conversation can begin.")
else:
    failing = [l for l in r.stdout.splitlines() if l.startswith("FAILED")]
    print("\nNot there yet. The next rule to restore:")
    print(" ", failing[0] if failing else "(scroll the output above)")

## Part C · Lab 2: the Earnings Analysis Engine

From an earnings-call transcript to a structured, **evidence-verified** note. The transcript (`session-03-debugging/data/transcript_meridian_q2_fy2026.txt`) is synthetic and the company fictional, so the model cannot rely on memorized knowledge; every claim must come from the document.

The plumbing is imported from the lab starter: a schema that **forces every claim to carry a verbatim quote**, and an `analyze` function that calls Claude with the transcript. **Your work is the trust layer**: the code that checks whether those quotes are real.

In [ ]:
sys.path.insert(0, str(ROOT / "session-03-debugging" / "lab"))
import earnings_starter as engine
from toolkit import llm            # for llm.show(): renders long output readably
from earnings_starter import EARNINGS_SCHEMA, analyze, DEFAULT_TRANSCRIPT

# The grounding rules, applied at the source (the same discipline as Session 1):
engine.SYSTEM = """You are a buy-side equity analyst preparing an internal note.
- Use ONLY the transcript provided. No outside knowledge, no memory of other companies.
- Every evidence_quote must be VERBATIM from the transcript; each will be machine-checked.
- If the transcript does not support a claim, do not make it.
- Separate management's framing from fact; note what guidance excludes."""

transcript = DEFAULT_TRANSCRIPT.read_text()
print(f"{len(transcript.split())} words. Speakers: CEO, CFO, five analysts. The transcript contains 7 planted red flags.")
print("Schema sections requiring verbatim evidence:", "key_themes, risks, red_flags")

### Exercise 2: verify_evidence, the fabrication detector

Every claim the model makes carries a verbatim quote. You check each quote against the source **in plain Python**. Normalize both sides first (collapse whitespace, lowercase, straighten curly quotes) so that formatting differences cannot cause false negatives. Set `item["verified"]` on every item in `key_themes`, `risks`, `red_flags`, and store totals in `analysis["_verification"]`.

In [ ]:
import re

def _normalize(text: str) -> str:
    """Whitespace-collapse + casefold + straighten curly quotes. GIVEN - it's
    plumbing; YOUR work is the verification logic below."""
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    return re.sub(r"\s+", " ", text).casefold().strip()

def verify_evidence(analysis: dict, transcript: str) -> dict:
### START CODE HERE ###
    haystack = _normalize(None)                        # normalize which text?
    checked = failed = 0
    for section in ("key_themes", "risks", "red_flags"):
        for item in analysis.get(section, []):
            quote = item.get("evidence_quote", "")
            item["verified"] = bool(quote) and None in haystack   # hint: the NORMALIZED quote
            checked += 1
            failed += 0 if item["verified"] else 1
    analysis["_verification"] = {"quotes_checked": None, "quotes_failed": None}
### END CODE HERE ###
    return analysis

print("defined - now catch a fabrication:")

In [ ]:
# ✅ self-check: run me. The canned dry-run analysis hides ONE deliberately
# fabricated quote. If your verify_evidence works, it catches exactly that one.
analysis = verify_evidence(analyze(transcript, dry_run=True), transcript)
v = analysis["_verification"]
print(f"quotes checked: {v['quotes_checked']}, failed: {v['quotes_failed']}")
assert v["quotes_checked"] >= 10, "check key_themes, risks AND red_flags"
assert v["quotes_failed"] == 1, "exactly ONE quote is fabricated - if 0, your matching is too loose; if >1, normalize better"
fake = [t for s in ("key_themes", "risks", "red_flags") for t in analysis[s] if not t["verified"]]
print("All checks passed ✅  Caught fabrication:", repr(fake[0]["evidence_quote"]))

**The same contract, in the industry's words.** The engine's schema is a raw JSON dict; in professional Python the standard way to declare and enforce such a contract is **Pydantic**: one class per object, one typed field per key. The cell below declares the analysis schema as Pydantic models and re-validates the model output through it — the same gate, now typed. Everything downstream could then use `analysis.red_flags[0].flag` instead of dictionary keys, and any malformed reply raises a precise `ValidationError` naming the offending field.

In [ ]:
from pydantic import BaseModel

class Theme(BaseModel):
    theme: str
    evidence_quote: str
    verified: bool | None = None

class Risk(BaseModel):
    risk: str
    severity: str
    evidence_quote: str
    verified: bool | None = None

class RedFlag(BaseModel):
    flag: str
    why_it_matters: str
    evidence_quote: str
    verified: bool | None = None

class EarningsAnalysis(BaseModel):
    overall_sentiment: str
    key_themes: list[Theme]
    risks: list[Risk]
    red_flags: list[RedFlag]

typed = EarningsAnalysis.model_validate(analysis)   # raises ValidationError on any shape violation
print(f"Validated: {len(typed.key_themes)} themes, {len(typed.risks)} risks, "
      f"{len(typed.red_flags)} red flags - all typed.")
print("First red flag, by attribute access:", typed.red_flags[0].flag)

That quote, a promised margin recovery, is entirely plausible and appears nowhere in the transcript. **Reading alone would likely have missed it; your code did not.** This is verification in practice.

**What to expect on a live run.** The cell below also runs the engine against the real API if you have a key. There the model writes its own quotes, and a current model usually quotes accurately, so you should expect **0 failures**. That is your checker working, not failing: it reports what it finds. The planted fabrication exists in the canned analysis precisely so that every student sees a catch at least once. In production the value of this layer is not that it fires often; it is that nothing reaches a committee unchecked.

In [ ]:
def memo_from(analysis: dict, source_name: str) -> str:
    """A compact analyst note: every claim printed with its verification mark."""
    lines = [f"# Meridian Semiconductor - Q2 FY2026 note (draft, from {source_name})",
             f"\n**Sentiment:** {analysis['overall_sentiment']} - {analysis['sentiment_rationale']}\n"]
    for section, key in [("Key themes", "theme"), ("Risks", "risk"), ("Red flags", "flag")]:
        lines.append(f"## {section}\n")
        for it in analysis[section.lower().replace(" ", "_")]:
            mark = "verified" if it.get("verified") else "NOT FOUND IN TRANSCRIPT"
            lines.append(f"- **{it[key]}** [{mark}]")
            lines.append(f"  - evidence: \"{it['evidence_quote'][:140]}\"")
    v = analysis["_verification"]
    lines.append(f"\n*Evidence check: {v['quotes_checked'] - v['quotes_failed']}/{v['quotes_checked']} quotes verified against the source.*")
    return "\n".join(lines)

OUTD = ROOT / "outputs"; OUTD.mkdir(exist_ok=True)
(OUTD / "meridian_earnings_memo.md").write_text(memo_from(analysis, "dry-run"))
print("outputs/meridian_earnings_memo.md written (dry-run).\n")

if HAS_KEY:
    live = verify_evidence(analyze(transcript, dry_run=False), transcript)
    lv = live["_verification"]
    (OUTD / "meridian_earnings_memo.md").write_text(memo_from(live, "LIVE"))
    print(f"LIVE run: {lv['quotes_checked'] - lv['quotes_failed']}/{lv['quotes_checked']} quotes verified.")
    print("0 failures is the expected result with a current model: the checker reports, it does not accuse.\n")
    llm.show(memo_from(live, "LIVE"), title="Your memo, every claim marked")
else:
    print("No API key - the dry-run memo still demonstrates the whole pipeline.")

## Deliverable checklist

- [ ] All 7 tests pass and `broken_dcf.py` values Meridian at **$75.61 vs $62 market**, with every fix made by Claude and reviewed by you
- [ ] You can name, in one sentence each, two financial errors Claude explained to you
- [ ] Your `verify_evidence` catches **exactly 1** fabricated quote in dry-run
- [ ] `outputs/meridian_earnings_memo.md` generated (live if you have a key) and committed to your repo

**Next:** `04-workflows-edgar.ipynb`, where we stop pasting context by hand and start fetching it programmatically, from live SEC filings.